# CS 4710 OSRS Protocol-Validation Dataset Audit

This notebook audits the completed, private 18-image protocol-validation mini-pilot. It prefers the canonical accepted manifest and falls back to the three accepted session manifests only when the canonical file is absent. It validates schema, acceptance and privacy-review state, class/session balance, identifiers, reviewed-image geometry, metadata, and exact SHA-256 duplicates before showing local reviewed images grouped by session.

**Current gate:** all 18 screenshots are collected, owner-reviewed, included, and `redaction_checked=true`; deterministic protocol validation passed. This set validates the collection and review process, not final classifier feasibility or model quality. No split assignment, feature extraction, model training, model evaluation, live capture, gameplay control, multi-game implementation, or Gamma runtime integration has occurred. The tracked notebook remains output-free.

In [ ]:
from __future__ import annotations

import csv
import hashlib
import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
from PIL import Image

PROJECT_ROOT = Path("..")
DATA_ROOT = PROJECT_ROOT / "data"
CANONICAL_MANIFEST = DATA_ROOT / "manifests" / "protocol_validation_18.csv"
PROTOCOL_AUDIT = DATA_ROOT / "manifests" / "protocol_validation_18_audit.json"
FALLBACK_MANIFESTS = {
    "session_001": DATA_ROOT / "manifests" / "session_001.csv",
    "session_002": DATA_ROOT / "manifests" / "session_002.csv",
    "session_003": DATA_ROOT / "manifests" / "session_003.csv",
}

if CANONICAL_MANIFEST.is_file():
    manifest_sources = {"protocol_validation_18": CANONICAL_MANIFEST}
    loaded_source = "canonical accepted protocol_validation_18.csv"
else:
    missing = [session_id for session_id, path in FALLBACK_MANIFESTS.items() if not path.is_file()]
    if missing:
        raise FileNotFoundError(f"Missing accepted fallback manifests for: {', '.join(missing)}")
    manifest_sources = FALLBACK_MANIFESTS
    loaded_source = "three accepted session manifests (canonical manifest absent)"

print(f"Loaded source: {loaded_source}")
{key: path.as_posix() for key, path in manifest_sources.items()}

## Accepted-manifest validation

The loaded source must use the exact approved 22-column schema and resolve to the same 18 accepted rows. Every row must be owner-captured, included, privacy-checked, and still `split=unassigned`.

In [ ]:
EXPECTED_HEADERS = (
    "image_id", "relative_image_path", "label", "capture_session_id",
    "source_group_id", "inclusion_status", "exclusion_reason",
    "resolution", "aspect_ratio", "client_type", "display_mode",
    "ui_layout_or_scale", "crop_policy", "camera_angle_bucket",
    "zoom_bucket", "location_or_activity_bucket", "overlay_notes",
    "redaction_method", "redaction_checked", "source_consent",
    "split", "notes",
)
APPROVED_LABELS = {
    "normal_gameplay", "bank_open", "dialogue_or_choice_open",
    "shop_open", "grand_exchange_open", "unknown_other",
}
SESSION_IDS = ("session_001", "session_002", "session_003")
EXPECTED_IDS = [f"img_{index:04d}" for index in range(1, 19)]
EXPECTED_GROUPS = [f"source_group_{index:04d}" for index in range(1, 19)]

all_rows = []
for source_name, manifest_path in manifest_sources.items():
    with manifest_path.open("r", encoding="utf-8", newline="") as handle:
        reader = csv.DictReader(handle)
        headers = tuple(reader.fieldnames or ())
        rows = list(reader)
    assert headers == EXPECTED_HEADERS
    assert len(headers) == len(set(headers)) == 22
    assert len(rows) == (18 if source_name == "protocol_validation_18" else 6)
    all_rows.extend(rows)

assert len(all_rows) == 18
assert [row["image_id"] for row in all_rows] == EXPECTED_IDS
assert [row["source_group_id"] for row in all_rows] == EXPECTED_GROUPS
assert len({row["relative_image_path"] for row in all_rows}) == 18
assert all(not Path(row["relative_image_path"]).is_absolute() for row in all_rows)
assert {row["label"] for row in all_rows} == APPROVED_LABELS
assert {row["inclusion_status"] for row in all_rows} == {"included"}
assert {row["redaction_checked"] for row in all_rows} == {"true"}
assert {row["source_consent"] for row in all_rows} == {"owner_captured"}
assert {row["split"] for row in all_rows} == {"unassigned"}
assert all(row["exclusion_reason"] == "" for row in all_rows)

session_rows = {
    session_id: [row for row in all_rows if row["capture_session_id"] == session_id]
    for session_id in SESSION_IDS
}
assert all(len(rows) == 6 for rows in session_rows.values())
accepted_rows = [row for row in all_rows if row["inclusion_status"] == "included"]
assert len(accepted_rows) == 18
print("Accepted-manifest validation passed: 18 included and privacy-checked rows; all 18 splits remain unassigned.")

## Per-session and class balance

In [ ]:
session_row_counts = {session_id: len(rows) for session_id, rows in session_rows.items()}
session_class_counts = {
    session_id: Counter(row["label"] for row in rows)
    for session_id, rows in session_rows.items()
}
combined_class_counts = Counter(row["label"] for row in all_rows)
session_label_pairs = Counter(
    (row["capture_session_id"], row["label"])
    for row in all_rows
)

assert session_row_counts == {"session_001": 6, "session_002": 6, "session_003": 6}
assert combined_class_counts == Counter({label: 3 for label in APPROVED_LABELS})
assert all(
    counts == Counter({label: 1 for label in APPROVED_LABELS})
    for counts in session_class_counts.values()
)
assert all(
    session_label_pairs[(session_id, label)] == 1
    for session_id in SESSION_IDS
    for label in APPROVED_LABELS
)
{
    "rows_by_session": session_row_counts,
    "classes_by_session": {
        key: dict(sorted(value.items())) for key, value in session_class_counts.items()
    },
    "combined_class_counts": dict(sorted(combined_class_counts.items())),
    "accepted_rows": len(accepted_rows),
    "redaction_checked_rows": sum(row["redaction_checked"] == "true" for row in all_rows),
    "unassigned_split_rows": sum(row["split"] == "unassigned" for row in all_rows),
}

## Reviewed-image and protocol-audit validation

These checks load only the local ignored reviewed files. Exact SHA-256 equality is a narrow duplicate check; it does not replace later perceptual-similarity and manual source-group review. When the canonical manifest is loaded, its private protocol audit is also checked.

In [ ]:
image_records = []
for row in all_rows:
    image_path = DATA_ROOT / row["relative_image_path"]
    assert image_path.is_file()
    assert image_path.stem == row["image_id"]
    digest = hashlib.sha256(image_path.read_bytes()).hexdigest()
    with Image.open(image_path) as image:
        image.load()
        image_records.append({
            "session_id": row["capture_session_id"],
            "image_id": row["image_id"],
            "label": row["label"],
            "dimensions": image.size,
            "mode": image.mode,
            "format": image.format,
            "metadata_keys": tuple(sorted(image.info)),
            "sha256": digest,
        })

assert len(image_records) == 18
assert len({record["image_id"] for record in image_records}) == 18
assert all(record["format"] == "PNG" for record in image_records)
assert all(record["dimensions"] == (2287, 1369) for record in image_records)
assert all(record["mode"] in {"RGB", "RGBA"} for record in image_records)
assert all(not record["metadata_keys"] for record in image_records)
assert len({record["sha256"] for record in image_records}) == 18

if CANONICAL_MANIFEST.is_file():
    assert PROTOCOL_AUDIT.is_file()
    protocol_audit = json.loads(PROTOCOL_AUDIT.read_text(encoding="utf-8"))
    manifest_digest = hashlib.sha256(CANONICAL_MANIFEST.read_bytes()).hexdigest()
    assert protocol_audit["manifest_sha256"] == manifest_digest
    assert protocol_audit["reviewed_image_sha256"] == {
        record["image_id"]: record["sha256"] for record in image_records
    }
    assert protocol_audit["exact_source_duplicate_count"] == 0
    assert protocol_audit["exact_reviewed_duplicate_count"] == 0
    assert all(protocol_audit["protocol_checks"].values())
    assert protocol_audit["protocol_validation_result"] == "passed"

print("All 18 reviewed PNGs have valid geometry, modes, and empty PNG metadata; no exact duplicates were found; deterministic protocol validation passed.")
[(record["session_id"], record["image_id"], record["dimensions"], record["mode"]) for record in image_records]

## Local visual review grouped by session

All three columns show accepted, owner-reviewed sessions. Displayed copies do not modify reviewed files or save private image output into this notebook.

In [ ]:
figure, axes = plt.subplots(6, 3, figsize=(27, 30), constrained_layout=True)
for column, session_id in enumerate(SESSION_IDS):
    for row_index, row in enumerate(session_rows[session_id]):
        axis = axes[row_index, column]
        image_path = DATA_ROOT / row["relative_image_path"]
        with Image.open(image_path) as image:
            image.load()
            axis.imshow(image)
        axis.set_title(f"{session_id}: {row['image_id']} — {row['label']}")
        axis.axis("off")
figure.suptitle("Accepted session_001 | Accepted session_002 | Accepted session_003", fontsize=18)
plt.show()

## Completed protocol-validation gate and next-stage status

- Protocol-validation collection is complete: three capture sessions and 18 screenshots.
- Owner review is complete: all 18 rows are included and all 18 have `redaction_checked=true`.
- Every session contributes six rows and exactly one example of each label; every label has three examples total.
- All 18 image IDs, reviewed paths, source groups, and reviewed hashes are unique; the protocol audit also confirms no exact source duplicates.
- All 18 splits remain `unassigned`.
- Deterministic protocol validation passed. The set validates manual collection, fixed geometry/cropping, label and manifest conventions, provenance, privacy review, and accepted-manifest generation.
- This mini-pilot is not the intended final feasibility-model dataset and does not establish classifier accuracy, generalization, gameplay understanding, multi-game support, or runtime readiness.
- The minimum feasibility target is 60 accepted images: 10 per class. The current count is 18, so 42 additional accepted images are required (seven per class).
- No feature extraction, model training, model evaluation, live capture, gameplay control, multi-game implementation, or Gamma runtime integration has occurred.